### Setting Base Directory

In [1]:
BASE_DIR = r"D:\dataset"

### Importing Necessary Libraries

In [2]:
# All paths and settings
import os
import glob
import math
import time
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

### Config & GPU Check

In [ ]:
TRAIN_HR = os.path.join(BASE_DIR, "DF2K_sampled", "train", "HR")
TRAIN_LR = os.path.join(BASE_DIR, "DF2K_sampled", "train", "LR_X2")
VAL_HR = os.path.join(BASE_DIR, "DF2K_sampled", "valid", "HR")
VAL_LR = os.path.join(BASE_DIR, "DF2K_sampled", "valid", "LR_X2")
SAVE_DIR = os.path.join(BASE_DIR, "results", "unet")
os.makedirs(SAVE_DIR, exist_ok=True)

SCALE_FACTOR = 2
BATCH_SIZE = 4
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
HR_PATCH_SIZE = 512
LR_PATCH_SIZE = HR_PATCH_SIZE // SCALE_FACTOR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Loading HR-LR patch pairs as PyTorch tensors

In [4]:
class SRDataset(Dataset):
    def __init__(self, hr_dir, lr_dir):
        self.hr_files = sorted(glob.glob(os.path.join(hr_dir, "*.png")))
        self.lr_files = sorted(glob.glob(os.path.join(lr_dir, "*.png")))
        assert len(self.hr_files) == len(self.lr_files), \
            f"Mismatch: {len(self.hr_files)} HR vs {len(self.lr_files)} LR"
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.hr_files)

    def __getitem__(self, idx):
        hr = self.to_tensor(Image.open(self.hr_files[idx]).convert("RGB"))
        lr = self.to_tensor(Image.open(self.lr_files[idx]).convert("RGB"))
        return lr, hr

train_dataset = SRDataset(TRAIN_HR, TRAIN_LR)
val_dataset = SRDataset(VAL_HR, VAL_LR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)} patches, {len(train_loader)} batches")
print(f"Valid: {len(val_dataset)} patches, {len(val_loader)} batches")

Train: 2500 patches, 625 batches
Valid: 100 patches, 25 batches


In [5]:
# Lightweight U-Net for x2 super resolution
# Bicubic upscales LR first, then U-Net refines the details with skip connections

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class SRUNet(nn.Module):
    def __init__(self, scale_factor=2, base_filters=32):
        super().__init__()
        self.scale_factor = scale_factor
        f = base_filters

        self.enc1 = ConvBlock(3, f)
        self.enc2 = ConvBlock(f, f * 2)
        self.enc3 = ConvBlock(f * 2, f * 4)
        self.bottleneck = ConvBlock(f * 4, f * 8)

        self.up3 = nn.ConvTranspose2d(f * 8, f * 4, 2, stride=2)
        self.dec3 = ConvBlock(f * 8, f * 4)
        self.up2 = nn.ConvTranspose2d(f * 4, f * 2, 2, stride=2)
        self.dec2 = ConvBlock(f * 4, f * 2)
        self.up1 = nn.ConvTranspose2d(f * 2, f, 2, stride=2)
        self.dec1 = ConvBlock(f * 2, f)

        self.final = nn.Conv2d(f, 3, 1)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x_up = F.interpolate(x, scale_factor=self.scale_factor, mode='bicubic', align_corners=False)

        e1 = self.enc1(x_up)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))

        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return (self.final(d1) + x_up).clamp(0, 1)


model = SRUNet(scale_factor=2, base_filters=32).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

dummy = torch.randn(1, 3, 256, 256).to(device)
print(f"Input: {dummy.shape} -> Output: {model(dummy).shape}")
del dummy
torch.cuda.empty_cache() if torch.cuda.is_available() else None

Parameters: 1,928,483
Input: torch.Size([1, 3, 256, 256]) -> Output: torch.Size([1, 3, 512, 512])


In [6]:
# PSNR to measure quality, L1 loss for training, scheduler to reduce LR over time
def calc_psnr(pred, target):
    mse = F.mse_loss(pred, target)
    if mse == 0:
        return float('inf')
    return 10 * math.log10(1.0 / mse.item())

criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

In [ ]:
# Train the model, save the best one based on validation PSNR
train_losses, val_losses = [], []
train_psnrs, val_psnrs = [], []
best_val_psnr = 0

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()

    # Train
    model.train()
    t_loss, t_psnr = 0, 0
    for lr_imgs, hr_imgs in train_loader:
        lr_imgs, hr_imgs = lr_imgs.to(device), hr_imgs.to(device)
        output = model(lr_imgs)
        loss = criterion(output, hr_imgs)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
        t_psnr += calc_psnr(output.detach(), hr_imgs)

    # Validate
    model.eval()
    v_loss, v_psnr = 0, 0
    with torch.no_grad():
        for lr_imgs, hr_imgs in val_loader:
            lr_imgs, hr_imgs = lr_imgs.to(device), hr_imgs.to(device)
            output = model(lr_imgs)
            v_loss += criterion(output, hr_imgs).item()
            v_psnr += calc_psnr(output, hr_imgs)

    # Record metrics
    train_losses.append(t_loss / len(train_loader))
    val_losses.append(v_loss / len(val_loader))
    train_psnrs.append(t_psnr / len(train_loader))
    val_psnrs.append(v_psnr / len(val_loader))

    scheduler.step()

    # Save best model
    if val_psnrs[-1] > best_val_psnr:
        best_val_psnr = val_psnrs[-1]
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pth"))

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | "
              f"Train Loss: {train_losses[-1]:.4f} PSNR: {train_psnrs[-1]:.2f} | "
              f"Val Loss: {val_losses[-1]:.4f} PSNR: {val_psnrs[-1]:.2f} | "
              f"{time.time()-start:.1f}s")

torch.save(model.state_dict(), os.path.join(SAVE_DIR, "final_model.pth"))
print(f"\nDone! Best Val PSNR: {best_val_psnr:.2f} dB")

Epoch   1/50 | Train Loss: 0.0308 PSNR: 27.20 | Val Loss: 0.0190 PSNR: 29.81 | 4727.1s


In [ ]:
# Plot how loss decreased and PSNR improved over training
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label="Train")
axes[0].plot(val_losses, label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("L1 Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_psnrs, label="Train")
axes[1].plot(val_psnrs, label="Validation")
axes[1].set_title("PSNR")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("dB")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_curves.png"), dpi=150)
plt.show()

In [ ]:
# Zoom into center to clearly see the detail differences
lr, hr = val_dataset[0]
lr_input = lr.unsqueeze(0).to(device)

with torch.no_grad():
    sr = model(lr_input)

bicubic = F.interpolate(lr_input, scale_factor=SCALE_FACTOR, mode='bicubic', align_corners=False).clamp(0, 1)

top, left = 192, 192  # Center crop region

images = {
    "Bicubic": bicubic.squeeze(0).cpu(),
    "U-Net": sr.squeeze(0).cpu(),
    "HR Target": hr,
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, img) in zip(axes, images.items()):
    crop = img[:, top:top+128, left:left+128]
    ax.imshow(crop.permute(1, 2, 0).clamp(0, 1))
    ax.set_title(title, fontsize=14)
    ax.axis("off")

plt.suptitle("Zoomed Comparison (128x128 crop)", fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "zoomed_comparison.png"), dpi=150)
plt.show()

In [ ]:
# Calculate PSNR and SSIM for both bicubic baseline and U-Net on full validation set
# Run: pip install torchmetrics   (if not installed)
from torchmetrics.image import StructuralSimilarityIndexMeasure

ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

model.eval()
psnr_bic, psnr_model = 0, 0
ssim_bic, ssim_model = 0, 0

with torch.no_grad():
    for lr_imgs, hr_imgs in val_loader:
        lr_imgs, hr_imgs = lr_imgs.to(device), hr_imgs.to(device)
        bicubic = F.interpolate(lr_imgs, scale_factor=SCALE_FACTOR, mode='bicubic', align_corners=False).clamp(0, 1)
        sr = model(lr_imgs)

        psnr_bic += calc_psnr(bicubic, hr_imgs)
        psnr_model += calc_psnr(sr, hr_imgs)
        ssim_bic += ssim_metric(bicubic, hr_imgs).item()
        ssim_model += ssim_metric(sr, hr_imgs).item()

n = len(val_loader)
print(f"{'='*50}")
print(f"VALIDATION METRICS")
print(f"{'='*50}")
print(f"{'Method':<15} {'PSNR (dB)':>12} {'SSIM':>12}")
print(f"{'-'*50}")
print(f"{'Bicubic':<15} {psnr_bic/n:>12.2f} {ssim_bic/n:>12.4f}")
print(f"{'U-Net':<15} {psnr_model/n:>12.2f} {ssim_model/n:>12.4f}")
print(f"{'='*50}")